In [2]:
# pip install plotly dash
import dash
from dash import dcc, html, Input, Output
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
import pandas as pd
import yfinance as yf
from scipy import stats
import time
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
#  Load & prepare data (reuse from before)
# ─────────────────────────────────────────────
STOCKS = ["AAPL", "MSFT", "JPM", "JNJ", "XOM",
          "AMZN", "TSLA", "GS", "PFE", "SPY"]

SECTORS = {
    "AAPL": "Technology", "MSFT": "Technology",
    "JPM":  "Finance",    "GS":   "Finance",
    "JNJ":  "Healthcare", "PFE":  "Pharma",
    "XOM":  "Energy",     "AMZN": "Consumer",
    "TSLA": "Automotive", "SPY":  "Index"
}

print("Fetching data...")
all_data = {}
for ticker in STOCKS:
    try:
        df = yf.download(ticker, start="2019-01-01",
                         end="2024-01-01", auto_adjust=True, progress=False)
        all_data[ticker] = df
        time.sleep(0.5)
    except:
        pass

prices = pd.DataFrame({t: d["Close"].squeeze() for t, d in all_data.items()}).dropna()
log_returns = np.log(prices / prices.shift(1)).dropna()

weights        = np.array([1 / len(STOCKS)] * len(STOCKS))
port_returns   = log_returns @ weights
mean_ret       = port_returns.mean()
std_ret        = port_returns.std()

# Pre-run Monte Carlo
N_SIMS, N_DAYS, INIT = 10_000, 252, 100_000
np.random.seed(42)
rand_rets    = np.random.normal(mean_ret, std_ret, (N_DAYS, N_SIMS))
cum_rets     = np.cumprod(1 + rand_rets, axis=0)
port_paths   = INIT * cum_rets
final_vals   = port_paths[-1, :]

print("Data ready. Starting dashboard...")

# ─────────────────────────────────────────────
#  Dash App
# ─────────────────────────────────────────────
app = dash.Dash(__name__)
app.title = "AlphaPulse — Investment Risk Monitor"

DARK = "#0d1117"
CARD = "#161b22"
ACCENT = "#58a6ff"

app.layout = html.Div(style={"backgroundColor": DARK, "minHeight": "100vh",
                              "fontFamily": "Arial, sans-serif", "padding": "20px"}, children=[

    # Header
    html.H1("AlphaPulse — Investment Risk & Volatility Monitor",
            style={"color": ACCENT, "textAlign": "center", "marginBottom": "8px"}),
    html.P("Real-time What-If scenario analysis | Monte Carlo VaR | Correlation Intelligence",
           style={"color": "#8b949e", "textAlign": "center", "marginBottom": "30px"}),

    # ── WHAT-IF CONTROLS (the Tableau "Parameters" equivalent) ──
    html.Div(style={"backgroundColor": CARD, "borderRadius": "10px",
                    "padding": "20px", "marginBottom": "20px",
                    "border": "1px solid #30363d"}, children=[

        html.H3("What-If Scenario Controls", style={"color": "#f0f6fc", "marginBottom": "15px"}),

        html.Div(style={"display": "grid", "gridTemplateColumns": "1fr 1fr 1fr",
                        "gap": "20px"}, children=[

            # Control 1: Tech sector shock
            html.Div([
                html.Label("Tech Sector Shock (%)",
                           style={"color": "#8b949e", "fontSize": "13px"}),
                dcc.Slider(id="tech-shock", min=-50, max=20, step=1, value=0,
                           marks={-50: "-50%", -25: "-25%", 0: "0%", 20: "+20%"},
                           tooltip={"always_visible": True, "placement": "bottom"}),
            ]),

            # Control 2: Finance sector shock
            html.Div([
                html.Label("Finance Sector Shock (%)",
                           style={"color": "#8b949e", "fontSize": "13px"}),
                dcc.Slider(id="finance-shock", min=-50, max=20, step=1, value=0,
                           marks={-50: "-50%", -25: "-25%", 0: "0%", 20: "+20%"},
                           tooltip={"always_visible": True, "placement": "bottom"}),
            ]),

            # Control 3: Portfolio investment amount
            html.Div([
                html.Label("Initial Investment ($)",
                           style={"color": "#8b949e", "fontSize": "13px"}),
                dcc.Slider(id="init-value", min=10000, max=500000,
                           step=10000, value=100000,
                           marks={10000: "$10K", 250000: "$250K", 500000: "$500K"},
                           tooltip={"always_visible": True, "placement": "bottom"}),
            ]),
        ]),
    ]),

    # ── LIVE METRIC CARDS ──
    html.Div(id="metric-cards",
             style={"display": "grid", "gridTemplateColumns": "repeat(4, 1fr)",
                    "gap": "15px", "marginBottom": "20px"}),

    # ── ROW 1: Monte Carlo + Distribution ──
    html.Div(style={"display": "grid", "gridTemplateColumns": "1fr 1fr",
                    "gap": "20px", "marginBottom": "20px"}, children=[
        html.Div(dcc.Graph(id="monte-carlo-chart"), style={
            "backgroundColor": CARD, "borderRadius": "10px",
            "border": "1px solid #30363d"}),
        html.Div(dcc.Graph(id="distribution-chart"), style={
            "backgroundColor": CARD, "borderRadius": "10px",
            "border": "1px solid #30363d"}),
    ]),

    # ── ROW 2: Correlation Heatmap + Sector Impact ──
    html.Div(style={"display": "grid", "gridTemplateColumns": "1fr 1fr",
                    "gap": "20px", "marginBottom": "20px"}, children=[
        html.Div(dcc.Graph(id="correlation-heatmap"), style={
            "backgroundColor": CARD, "borderRadius": "10px",
            "border": "1px solid #30363d"}),
        html.Div(dcc.Graph(id="sector-impact-chart"), style={
            "backgroundColor": CARD, "borderRadius": "10px",
            "border": "1px solid #30363d"}),
    ]),

    # ── ROW 3: Price lines + Rolling Volatility ──
    html.Div(style={"display": "grid", "gridTemplateColumns": "1fr 1fr",
                    "gap": "20px"}, children=[
        html.Div(dcc.Graph(id="price-chart"), style={
            "backgroundColor": CARD, "borderRadius": "10px",
            "border": "1px solid #30363d"}),
        html.Div(dcc.Graph(id="volatility-chart"), style={
            "backgroundColor": CARD, "borderRadius": "10px",
            "border": "1px solid #30363d"}),
    ]),
])


# ─────────────────────────────────────────────
#  CALLBACK — updates ALL charts instantly
#  when any slider moves (the "What-If" engine)
# ─────────────────────────────────────────────
@app.callback(
    Output("metric-cards",        "children"),
    Output("monte-carlo-chart",   "figure"),
    Output("distribution-chart",  "figure"),
    Output("correlation-heatmap", "figure"),
    Output("sector-impact-chart", "figure"),
    Output("price-chart",         "figure"),
    Output("volatility-chart",    "figure"),
    Input("tech-shock",     "value"),
    Input("finance-shock",  "value"),
    Input("init-value",     "value"),
)
def update_all(tech_shock, finance_shock, init_value):

    # ── Apply What-If shocks to returns ──
    shocked_returns = log_returns.copy()
    for ticker, sector in SECTORS.items():
        if sector == "Technology" and ticker in shocked_returns.columns:
            shocked_returns[ticker] += (tech_shock / 100) / 252
        if sector == "Finance" and ticker in shocked_returns.columns:
            shocked_returns[ticker] += (finance_shock / 100) / 252

    port_ret_shocked = shocked_returns @ weights
    mean_s = port_ret_shocked.mean()
    std_s  = port_ret_shocked.std()

    # Re-run Monte Carlo with shocked params
    np.random.seed(42)
    r = np.random.normal(mean_s, std_s, (N_DAYS, N_SIMS))
    paths = init_value * np.cumprod(1 + r, axis=0)
    fvals = paths[-1, :]

    var_95  = init_value - np.percentile(fvals, 5)
    mean_fv = np.mean(fvals)
    ann_vol = std_s * np.sqrt(252)
    ann_ret = mean_s * 252

    # ── Metric Cards ──
    def card(label, value, color):
        return html.Div(style={"backgroundColor": CARD, "borderRadius": "10px",
                                "padding": "18px", "border": f"1px solid {color}",
                                "textAlign": "center"}, children=[
            html.P(label, style={"color": "#8b949e", "margin": "0",
                                  "fontSize": "13px"}),
            html.H3(value, style={"color": color, "margin": "6px 0 0"}),
        ])

    cards = [
        card("Expected Value (1yr)", f"${mean_fv:,.0f}", "#58a6ff"),
        card("Value at Risk (95%)",  f"${var_95:,.0f}",  "#f85149"),
        card("Annual Volatility",    f"{ann_vol:.1%}",   "#e3b341"),
        card("Expected Return",      f"{ann_ret:.1%}",   "#3fb950"),
    ]

    layout_dark = dict(
        paper_bgcolor=CARD, plot_bgcolor=DARK,
        font=dict(color="#c9d1d9"),
        margin=dict(l=40, r=20, t=50, b=40),
        xaxis=dict(gridcolor="#21262d"),
        yaxis=dict(gridcolor="#21262d"),
    )

    # ── Chart 1: Monte Carlo paths ──
    fig_mc = go.Figure()
    for i in range(0, 300, 3):   # plot 100 paths
        fig_mc.add_trace(go.Scatter(
            y=paths[:, i], mode="lines",
            line=dict(color="#58a6ff", width=0.4),
            opacity=0.12, showlegend=False))
    for pct, col, name in [(95,"#3fb950","95th %ile"),
                            (50,"#e3b341","Median"),
                            (5, "#f85149","5th %ile (VaR)")]:
        fig_mc.add_trace(go.Scatter(
            y=np.percentile(paths, pct, axis=1), mode="lines",
            line=dict(color=col, width=2.5), name=name))
    fig_mc.add_hline(y=init_value, line_dash="dash",
                     line_color="white", opacity=0.5,
                     annotation_text="Start")
    fig_mc.update_layout(**layout_dark,
        title="Monte Carlo — 10,000 Simulated Paths",
        yaxis_tickprefix="$", yaxis_tickformat=",")

    # ── Chart 2: Final value distribution ──
    fig_dist = go.Figure()
    fig_dist.add_trace(go.Histogram(
        x=fvals, nbinsx=80, marker_color="#58a6ff",
        opacity=0.8, name="Simulations"))
    fig_dist.add_vline(x=init_value, line_dash="dash",
                       line_color="white", annotation_text="Start")
    fig_dist.add_vline(x=np.percentile(fvals, 5), line_color="#f85149",
                       line_width=2, annotation_text="VaR 95%")
    fig_dist.add_vline(x=mean_fv, line_color="#e3b341",
                       line_width=2, annotation_text="Mean")
    skew = stats.skew(fvals)
    kurt = stats.kurtosis(fvals)
    fig_dist.update_layout(**layout_dark,
        title=f"Final Value Distribution | Skew: {skew:.2f} | Kurtosis: {kurt:.2f}",
        xaxis_tickprefix="$", xaxis_tickformat=",")

    # ── Chart 3: Correlation heatmap ──
    corr = log_returns.corr()
    fig_corr = go.Figure(go.Heatmap(
        z=corr.values, x=corr.columns, y=corr.index,
        colorscale="RdYlGn", zmid=0,
        text=np.round(corr.values, 2),
        texttemplate="%{text}", textfont=dict(size=10),
        colorbar=dict(title="Correlation")))
    fig_corr.update_layout(**layout_dark,
        title="Correlation Heatmap — All Stocks")

    # ── Chart 4: Sector What-If impact ──
    sector_shocks = {
        "Technology": tech_shock,
        "Finance":    finance_shock,
        "Healthcare": 0, "Energy": 0,
        "Consumer": 0, "Pharma": 0,
        "Automotive": 0, "Index": 0
    }
    colors = ["#f85149" if v < 0 else "#3fb950"
              for v in sector_shocks.values()]
    fig_sector = go.Figure(go.Bar(
        x=list(sector_shocks.keys()),
        y=list(sector_shocks.values()),
        marker_color=colors,
        text=[f"{v:+d}%" for v in sector_shocks.values()],
        textposition="outside"))
    fig_sector.add_hline(y=0, line_color="white", opacity=0.3)
    fig_sector.update_layout(**layout_dark,
        title="What-If: Sector Shock Applied",
        yaxis_title="Shock (%)")

    # ── Chart 5: Normalised price lines ──
    norm_prices = prices / prices.iloc[0] * 100
    fig_price = go.Figure()
    for col in norm_prices.columns:
        fig_price.add_trace(go.Scatter(
            x=norm_prices.index, y=norm_prices[col],
            mode="lines", name=col, line=dict(width=1.5)))
    fig_price.update_layout(**layout_dark,
        title="Normalised Price (Base=100)",
        yaxis_title="Indexed Value")

    # ── Chart 6: Rolling 30-day volatility ──
    rolling_vol = log_returns.rolling(30).std() * np.sqrt(252)
    port_vol    = (shocked_returns @ weights).rolling(30).std() * np.sqrt(252)
    fig_vol = go.Figure()
    for col in rolling_vol.columns:
        fig_vol.add_trace(go.Scatter(
            x=rolling_vol.index, y=rolling_vol[col],
            mode="lines", name=col,
            line=dict(width=1), opacity=0.6))
    fig_vol.add_trace(go.Scatter(
        x=port_vol.index, y=port_vol,
        mode="lines", name="Portfolio",
        line=dict(color="white", width=3)))
    fig_vol.update_layout(**layout_dark,
        title="30-Day Rolling Volatility (Annualised)",
        yaxis_tickformat=".0%")

    return cards, fig_mc, fig_dist, fig_corr, fig_sector, fig_price, fig_vol


# ─────────────────────────────────────────────
#  Run
# ─────────────────────────────────────────────
if __name__ == "__main__":
    app.run(debug=False, port=8050)

Fetching data...
Data ready. Starting dashboard...
